In [1]:
# kernel: thesis_clean4
#https://github.com/scaomath/fourier_neural_operator/blob/master/fourier_1d.py
import matplotlib.pyplot as plt
import pandas as pd
#from neuralop.models import FNO
from sklearn.preprocessing import OneHotEncoder
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.nn.functional as F
from torch.nn.parameter import Parameter
import math
from sklearn.metrics import f1_score

In [2]:
import torch
print(torch.__version__)

2.5.1+cu121


In [12]:
#data = pyscrew.get_data(scenario="s02", handle_duplicates="first", handle_missings="mean",force_download=True, target_length=800)
#df = pd.DataFrame(data)
#df.to_pickle("screw_data_s02-v2_identical-to-v1.pkl")



2026-06-27 16:10:02 - INFO - pyscrew.main - Starting data retrieval for scenario: s02 (surface-friction)
2026-06-27 16:10:02 - INFO - pyscrew.pipeline.loading - Using cache directory (absolute): c:\Users\Patrick\miniconda3\envs\thesis_clean4\Lib\site-packages\pyscrew\downloads
2026-06-27 16:10:02 - INFO - pyscrew.pipeline.loading - Beginning data extraction for scenario 's02_variations-in-surface-friction.zip' (force=True)
2026-06-27 16:10:02 - INFO - pyscrew.pipeline.loading - Downloading dataset 's02_variations-in-surface-friction.zip' from Zenodo URL: https://zenodo.org/records/16031381/files/s02_variations-in-surface-friction.zip?download=1


2026-06-27 16:10:28 - INFO - pyscrew.pipeline.loading - Download of 's02_variations-in-surface-friction.zip' completed (83,946,972 bytes). Beginning checksum verification...
2026-06-27 16:10:28 - INFO - pyscrew.pipeline.loading - Verifying MD5 checksum for 's02_variations-in-surface-friction.zip'...
2026-06-27 16:10:28 - INFO - pyscrew.pipeline.loading - Checksum verification successful for 's02_variations-in-surface-friction.zip' (MD5: 0bc948a6e8c6e83f72dbe36973131558)


2026-06-27 16:10:32 - INFO - pyscrew.pipeline.loading - Extracting archive 's02_variations-in-surface-friction.zip' to directory: c:\Users\Patrick\miniconda3\envs\thesis_clean4\Lib\site-packages\pyscrew\downloads\extracted\s02_variations-in-surface-friction
2026-06-27 16:10:38 - INFO - pyscrew.pipeline.loading - Extraction of 's02_variations-in-surface-friction.zip' completed successfully to: c:\Users\Patrick\miniconda3\envs\thesis_clean4\Lib\site-packages\pyscrew\downloads\extracted\s02_variations-in-surface-friction
2026-06-27 16:10:38 - INFO - pyscrew.core.dataset - Selected 12500 files
2026-06-27 16:10:49 - INFO - pyscrew.core.dataset - Successfully loaded 12500 screw runs
2026-06-27 16:10:49 - INFO - pyscrew.pipeline.processing - Adding input_logging transformer to pipeline
2026-06-27 16:10:49 - INFO - pyscrew.pipeline.processing - Adding step_unpacking transformer to pipeline
2026-06-27 16:10:49 - INFO - pyscrew.pipeline.processing - Adding duplicate handling with first
2026-06-2

2026-06-27 16:12:46 - INFO - pyscrew.pipeline.transformers.handle_missings - Completed missing interpolation using 'mean' method (interval=0.0012)
2026-06-27 16:12:46 - INFO - pyscrew.pipeline.transformers.handle_missings - Processed 12,500 series with 8,619,982 total points
2026-06-27 16:12:46 - INFO - pyscrew.pipeline.transformers.handle_missings - Found gaps - min: 0.0012s, max: 0.1128s, avg: 0.0013s
2026-06-27 16:12:46 - INFO - pyscrew.pipeline.transformers.handle_missings - Added 484,205 points (+5.62% of total)
2026-06-27 16:12:46 - INFO - pyscrew.pipeline.transformers.handle_missings - Average 38.7 points added per series


2026-06-27 16:12:46 - INFO - pyscrew.pipeline.transformers.handle_lengths - Starting to apply equal lengths.
2026-06-27 16:12:46 - INFO - pyscrew.pipeline.transformers.handle_lengths - - 'target_length' : 800
2026-06-27 16:12:46 - INFO - pyscrew.pipeline.transformers.handle_lengths - - 'padding_value' : 0.0
2026-06-27 16:12:46 - INFO - pyscrew.pipeline.transformers.handle_lengths - - 'padding_position' : post
2026-06-27 16:12:46 - INFO - pyscrew.pipeline.transformers.handle_lengths - - 'cutoff_position' : post
2026-06-27 16:12:48 - INFO - pyscrew.pipeline.transformers.handle_lengths - Finished applying equal lengths to the screw driving data.
2026-06-27 16:12:48 - INFO - pyscrew.pipeline.transformers.handle_lengths - - Total screw runs loaded:	12500
2026-06-27 16:12:48 - INFO - pyscrew.pipeline.transformers.handle_lengths - - Average change of length:	728.33 -> 800.00
2026-06-27 16:12:48 - INFO - pyscrew.pipeline.transformers.handle_lengths - - Total points before normalization:	9,104,

In [2]:
df = pd.read_pickle("screw_data_s02-v2_identical-to-v1.pkl")
df.head()

,time_values,torque_values,angle_values,gradient_values,step_values,class_values,workpiece_location,workpiece_usage,workpiece_result,scenario_condition,scenario_exception
0,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[0.067, 0.077, 0.126, 0.087, 0.089, 0.097, 0.0...","[0.5, 1.25, 2.25, 3.75, 5.0, 6.25, 7.5, 8.75, ...","[0.0, 0.0, 0.0298, 0.0214, 0.0064, 0.0023, -0....","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,0,OK,normal,0
1,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.005, 0.008, 0.061, 0.069, 0.104, 0.124, 0....","[0.0, 0.25, 0.75, 1.5, 2.5, 4.0, 5.25, 6.5, 7....","[0.0, 0.0, 0.0, 0.0, 0.0287, 0.0282, 0.0091, 0...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,right,0,OK,normal,0
2,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.003, 0.01, 0.039, 0.099, 0.102, 0.087, 0.0...","[0.0, 0.5, 1.25, 2.25, 3.5, 4.75, 6.0, 7.5, 8....","[0.0, 0.0, 0.0, 0.0196, 0.0223, 0.0211, 0.0096...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,1,OK,normal,0
3,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[0.081, 0.059, 0.12, 0.077, 0.043, 0.059, 0.06...","[0.75, 1.75, 2.75, 4.0, 5.25, 6.5, 7.75, 9.25,...","[0.0, 0.0, 0.0261, 0.0207, 0.0, -0.0036, -0.00...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,right,1,OK,normal,0
4,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.003, -0.0012, 0.0006, 0.0024, 0.0042, 0.00...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 1.5, 2.5, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.025...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,2,OK,normal,0


In [3]:
################################################################
#  1d fourier layer # auf 2 in channels angepasst
################################################################
class SpectralConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, modes1):
        super(SpectralConv1d, self).__init__()

        """
        1D Fourier layer. It does FFT, linear transform, and Inverse FFT.    
        """

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1  #Number of Fourier modes to multiply, at most floor(N/2) + 1

        self.scale = (1 / (in_channels*out_channels))
        self.weights1 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, dtype=torch.cfloat))

    # Complex multiplication
    def compl_mul1d(self, input, weights):
        # (batch, in_channel, x ), (in_channel, out_channel, x) -> (batch, out_channel, x)
        return torch.einsum("bix,iox->box", input, weights)

    def forward(self, x):
        batchsize = x.shape[0]
        #Compute Fourier coeffcients up to factor of e^(- something constant)
        x_ft = torch.fft.rfft(x)

        # Multiply relevant Fourier modes
        out_ft = torch.zeros(batchsize, self.out_channels, x.size(-1)//2 + 1,  device=x.device, dtype=torch.cfloat)
        out_ft[:, :, :self.modes1] = self.compl_mul1d(x_ft[:, :, :self.modes1], self.weights1)

        #Return to physical space
        x = torch.fft.irfft(out_ft, n=x.size(-1))
        return x

class FNO1d(nn.Module):
    def __init__(self, modes, width, n_classes, in_channels=1):
        super(FNO1d, self).__init__()

        """
        The overall network. It contains 4 layers of the Fourier layer.
        1. Lift the input to the desire channel dimension by self.fc0 .
        2. 4 layers of the integral operators u' = (W + K)(u).
            W defined by self.w; K defined by self.conv .
        3. Project from the channel space to the output space by self.fc1 and self.fc2 .
        
        input: the solution of the initial condition and location (a(x), x)
        input shape: (batchsize, x=s, c=2)
        output: the solution of a later timestep
        output shape: (batchsize, x=s, c=1)
        """

        self.modes1 = modes
        self.width = width
        self.padding = 2 # pad the domain if input is non-periodic
        #self.fc0 = nn.Linear(2, self.width) # input channel is 2: (a(x), x)
        self.fc0 = nn.Linear(in_channels + 1, self.width) # für 2 feature input
        
        self.conv0 = SpectralConv1d(self.width, self.width, self.modes1)
        self.conv1 = SpectralConv1d(self.width, self.width, self.modes1)
        self.conv2 = SpectralConv1d(self.width, self.width, self.modes1)
        self.conv3 = SpectralConv1d(self.width, self.width, self.modes1)
        self.w0 = nn.Conv1d(self.width, self.width, 1)
        self.w1 = nn.Conv1d(self.width, self.width, 1)
        self.w2 = nn.Conv1d(self.width, self.width, 1)
        self.w3 = nn.Conv1d(self.width, self.width, 1)

        self.fc1 = nn.Linear(self.width, 128)
        self.fc2 = nn.Linear(128, n_classes)

    def forward(self, x):
        grid = self.get_grid(x.shape, x.device)
        #x = torch.cat((x, grid), dim=-1)
        x = torch.cat((x, grid), dim=-1)
        x = self.fc0(x)
        x = x.permute(0, 2, 1)
        # x = F.pad(x, [0,self.padding]) # pad the domain if input is non-periodic

        x1 = self.conv0(x)
        x2 = self.w0(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv1(x)
        x2 = self.w1(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv2(x)
        x2 = self.w2(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv3(x)
        x2 = self.w3(x)
        x = x1 + x2

        # 1. Global Pooling: Average across the 1000 time steps
        # This collapses the sequence dimension
        x = torch.mean(x, dim=-1) # New shape: (batch, width)

        # x = x[..., :-self.padding] # pad the domain if input is non-periodic
        #x = x.permute(0, 2, 1)
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)
        return x

    def get_grid(self, shape, device):
        batchsize, size_x = shape[0], shape[1]
        gridx = torch.tensor(np.linspace(0, 1, size_x), dtype=torch.float)
        gridx = gridx.reshape(1, size_x, 1).repeat([batchsize, 1, 1])
        return gridx.to(device)

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#config 
ntrain = 1000
ntest = 100

sub = 2**3 #subsampling rate
h = 2**13 // sub #total grid size divided by the subsampling rate
s = h

batch_size = 20
learning_rate = 0.001

epochs = 500
step_size = 50
gamma = 0.5

modes = 16
width = 64
model = FNO1d(modes, width, n_classes=8, in_channels=2).to(device)

In [5]:
class EarlyStopper:
    def __init__(self, patience=1, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.min_validation_loss = float('inf')

    def early_stop(self, validation_loss):
        if validation_loss < self.min_validation_loss:
            self.min_validation_loss = validation_loss
            self.counter = 0
        elif validation_loss > (self.min_validation_loss + self.min_delta):
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False

# 3CV fold evaluation

In [6]:
from sklearn.metrics import f1_score
import math
torque = np.array(df['torque_values'].tolist())[..., np.newaxis]
angle = np.array(df['angle_values'].tolist())
time  = np.array(df['time_values'].tolist())

angle_rad = np.radians(angle)

dt = np.diff(time, axis=1) + 1e-8
dtheta = np.diff(angle_rad, axis=1)

omega = dtheta / dt

torque = np.array(df['torque_values'].tolist())

torque_trim = torque[:, 1:]  #auf -1 wieder anpassen da shapeomega

x_data = np.concatenate([torque_trim[..., None], omega[..., None]], axis=-1)
y_data = np.array(df['class_values'].tolist())
print("x_data shape:", x_data.shape)
print("transposed x_data shape:", x_data.shape)
le = LabelEncoder()
y_encoded = le.fit_transform(y_data)

X_train_full, X_test, y_train_full, y_test = train_test_split(x_data, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=81)

cv_scores = []

best_overall_model_state = None
best_overall_f1 = -np.inf


for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_full, y_train_full)):

    print(f"Fold {fold+1}")

    X_train_fold = X_train_full[train_idx]
    y_train_fold = y_train_full[train_idx]

    X_val_fold = X_train_full[val_idx]
    y_val_fold = y_train_full[val_idx]

    train_loader = DataLoader(TensorDataset(torch.tensor(X_train_fold, dtype=torch.float32),torch.tensor(y_train_fold, dtype=torch.long)),batch_size=32,shuffle=True)
    val_loader = DataLoader(TensorDataset(torch.tensor(X_val_fold, dtype=torch.float32),torch.tensor(y_val_fold, dtype=torch.long)),batch_size=32,shuffle=False)
    model = FNO1d(modes, width, n_classes=8, in_channels=2).to(device)
    model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

    earlystop = EarlyStopper(patience=7, min_delta=0.001)

    epochs = 50

    best_val_f1 = -np.inf
    best_model_state = None


    for epoch in range(epochs):

        model.train()
        train_loss = 0.0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)

        model.eval()
        val_loss = 0.0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for X_val_batch, y_val_batch in val_loader:
                X_val_batch = X_val_batch.to(device)
                y_val_batch = y_val_batch.to(device)

                outputs = model(X_val_batch)
                loss = criterion(outputs, y_val_batch)

                val_loss += loss.item()

                preds = torch.argmax(outputs, dim=1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y_val_batch.cpu().numpy())

        avg_val_loss = val_loss / len(val_loader)
        val_f1 = f1_score(all_labels, all_preds, average="macro")

        scheduler.step(avg_val_loss)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_state = model.state_dict()

        if earlystop.early_stop(avg_val_loss):
            print("Early stopping triggered")
            break

        print(f"Fold {fold+1}, Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Val F1: {val_f1:.4f}")

    cv_scores.append(best_val_f1)
    if best_val_f1 > best_overall_f1:
        best_overall_f1 = best_val_f1
        best_overall_model_state = best_model_state


print(f"CV f1 mean avg score: {np.mean(cv_scores):.4f}, std: {np.std(cv_scores):.4f}")

x_data shape: (12500, 799, 2)
transposed x_data shape: (12500, 799, 2)
Fold 1
Fold 1, Epoch 1/50, Train Loss: 1.9690, Val Loss: 1.9355, Val F1: 0.1140
Fold 1, Epoch 2/50, Train Loss: 1.9252, Val Loss: 1.9155, Val F1: 0.1391
Fold 1, Epoch 3/50, Train Loss: 1.8483, Val Loss: 1.7787, Val F1: 0.2280
Fold 1, Epoch 4/50, Train Loss: 1.7578, Val Loss: 1.7560, Val F1: 0.2108
Fold 1, Epoch 5/50, Train Loss: 1.7473, Val Loss: 1.7442, Val F1: 0.2364
Fold 1, Epoch 6/50, Train Loss: 1.7291, Val Loss: 1.7521, Val F1: 0.1776
Fold 1, Epoch 7/50, Train Loss: 1.7290, Val Loss: 1.7472, Val F1: 0.2272
Fold 1, Epoch 8/50, Train Loss: 1.7204, Val Loss: 1.7349, Val F1: 0.2301
Fold 1, Epoch 9/50, Train Loss: 1.7144, Val Loss: 1.7672, Val F1: 0.2096
Fold 1, Epoch 10/50, Train Loss: 1.6988, Val Loss: 1.7846, Val F1: 0.2063
Fold 1, Epoch 11/50, Train Loss: 1.6918, Val Loss: 1.7051, Val F1: 0.2368
Fold 1, Epoch 12/50, Train Loss: 1.6912, Val Loss: 1.7979, Val F1: 0.2113
Fold 1, Epoch 13/50, Train Loss: 1.6850, Va

In [7]:
model = FNO1d(modes, width, n_classes=8, in_channels=2).to(device)
final_model = model
final_model.to(device)
final_model.load_state_dict(best_overall_model_state)
final_model.eval()

test_loader = DataLoader(TensorDataset(torch.tensor(X_test, dtype=torch.float32),torch.tensor(y_test, dtype=torch.long)),batch_size=32,shuffle=False)

all_preds = []
all_labels = []
test_loss = 0.0

criterion = nn.CrossEntropyLoss()
with torch.no_grad():
    for X_test_batch, y_test_batch in test_loader:

        X_test_batch = X_test_batch.to(device)
        y_test_batch = y_test_batch.to(device)
        outputs = final_model(X_test_batch)
        loss = criterion(outputs, y_test_batch)
        test_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_test_batch.cpu().numpy())

test_loss = test_loss / len(test_loader)
test_f1 = f1_score(all_labels, all_preds, average="macro")
print(f"Final Test F1 Macro: {test_f1:.4f}")

Final Test F1 Macro: 0.3569


In [ ]:
import torch
print(torch.__version__)
